# Glacier evolution using OGGM

In [3]:
# basic
import os
from glob import glob
from tqdm import tqdm
import geopandas as gpd
import pandas as pd

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# oggm
from oggm import cfg, workflow, tasks, utils
from MBsandbox.help_func import melt_f_calib_geod_prep_inversion
from MBsandbox.flowline_TIModel import run_from_climate_data_TIModel, run_with_hydro_daily

# locals
from src.utils import load_config, assign_region
config = load_config()

## Initialization

In [ ]:
# OGGM configuration
cfg.initialize(logging_level='ERROR', future = True)
cfg.PATHS['working_dir']  = config["dir"]["pipeline"] + "/oggm/"

cfg.PARAMS['use_multiprocessing']  = True
cfg.PARAMS['baseline_climate']     = ''
cfg.PARAMS['prcp_scaling_factor']  = 1
cfg.PARAMS['hydro_month_sh']       = 1
cfg.PARAMS['hydro_month_nh']       = 1
cfg.PARAMS['border']               = 160 
cfg.PARAMS['geodetic_mb_period']   = '2000-01-01_2020-01-01'
cfg.PARAMS['store_model_geometry'] = True
cfg.PARAMS['continue_on_error']    = True
cfg.PARAMS['rgi_version'] = 62

In [ ]:
# Load RGI regions 16 and 17 + volume
rgi_16 = utils.get_rgi_region_file(16, version='62') # Low latitudes
rgi_17 = utils.get_rgi_region_file(17, version='62') # Southern Andes
rgi_16 = gpd.read_file(rgi_16).set_index("RGIId")
rgi_17 = gpd.read_file(rgi_17).set_index("RGIId")

rgi_ids = pd.concat([rgi_16, rgi_17])
rgi_ids = rgi_ids[(rgi_ids.CenLon < config["area"]["max_lon"]) & 
                   (rgi_ids.CenLon > config["area"]["min_lon"]) & 
                   (rgi_ids.CenLat < config["area"]["max_lat"])]
rgi_ids['region'] = rgi_ids.CenLat.apply(assign_region, regions=config["regions"])

volume = pd.read_csv("data/glacier/glacier_volume_frank_2026.csv", index_col=0)
rgi_ids = rgi_ids.join(volume)

6991.388116300001

## Historical run

In [ ]:
for region in config["regions"]:


    rgi_ids_subset = rgi_ids[rgi_ids['region'] == region]

    # init glacier directories
    gdirs = workflow.init_glacier_directories(rgi_ids_subset.index)

    # calibration using huggonet et al. 2021
    workflow.execute_entity_task(melt_f_calib_geod_prep_inversion, gdirs,
                                pf=1,                    # precipitation factor
                                mb_type="mb_real_daily", # use daily temperature values
                                grad_type="cte",         # constant lapse rate (-6.5)
                                climate_type="ERA5",     # climate type 
                                ye=2020,                 # last year
                                min_melt_f = 1.5*30.5,   # minimum melt factor that is allowed
                                max_melt_f = 17*30.5,    # maximum melt factor that is allowed
                                kwargs_for_TIModel_Sfc_Type = dict(t_melt  = 0, 
                                                                    t_solid = 0,
                                                                    t_liq   = 2));

    # ice thickness inversion 
    workflow.calibrate_inversion_from_consensus(gdirs, 
                                                apply_fs_on_mismatch=True,     # on mismatch, try to apply an arbitrary value of fs
                                                error_on_mismatch=False,       # sometimes the given bounds do not allow to find a zero mismatch: this will normally raise an error
                                                filter_inversion_output=True,   # apply terminus thickness filtering on the inversion output
                                                volume_m3_reference = rgi_ids_subset['volume_mk3'].sum()*1e9); 

    # ready to use 
    workflow.execute_entity_task(tasks.init_present_time_glacier, gdirs); 
    utils.compile_glacier_statistics(gdirs);  # save statistics

    # run
    workflow.execute_entity_task(run_with_hydro_daily, gdirs,
                                run_task = run_from_climate_data_TIModel, # any of the `run_*`` tasks in the MBSandbox.flowline_TIModel module.
                                ref_area_yr = 2000,                 # the hydrological output is computed over a reference area, specify here
                                ref_area_from_y0 = False,           # overwrite ref_area_yr to the first year of the timeseries
                                fixed_geometry_spinup_yr = 2000, 
                                ref_geometry_filesuffix = None,     # this kwarg allows to copy the reference area from a previous simulation
                                store_annual = True,               # whether to store annual outputs or only daily outputs
                                Testing = False,                    # if set to true, the 29th of February is set to nan values in non-leap years
                                ys=2000, ye=2025,                   # start and end years of the model run
                                climate_input_filesuffix="ERA5",    # filesuffix for the input climate file                        
                                climate_filename='climate_historical',
                                output_filesuffix='_historical',    # for the output file
                                init_model_filesuffix = None,       # if you want to start from a previous model run state
                                mb_type="mb_real_daily",            # use daily temperature values
                                grad_type="cte",                    # constant lapse rate (-6.5)
                                precipitation_factor=1,             # multiply a factor to the precipitation time series
                                melt_f='from_json');                # calibrated melt_f

    utils.compile_run_output(gdirs, input_filesuffix='_historical');

## Future runs

In [ ]:
scenarios  = glob(config["dir"]["data"] + "/cmip6/*.nc")

# future projections
for gcm in tqdm(scenarios):    

    rid = os.path.splitext(os.path.basename(gcm))[0].rsplit('_', 2)[0]
        
    workflow.execute_entity_task(run_with_hydro_daily, gdirs,
                                    run_task = run_from_climate_data_TIModel, # any of the `run_*`` tasks in the MBSandbox.flowline_TIModel module.
                                    ref_area_from_y0 = True,                  # overwrite ref_area_yr to the first year of the timeseries
                                    ref_geometry_filesuffix = "_historical",  # this kwarg allows to copy the reference area from a previous simulation
                                    store_annual = True,                     # whether to store annual outputs or only daily outputs
                                    Testing = False,                          # if set to true, the 29th of February is set to nan values in non-leap years
                                    ys=2025, ye=2100,                         # start and end years of the model run
                                    climate_input_filesuffix= rid,            # filesuffix for the input climate file                        
                                    output_filesuffix= "_"+  rid,             # for the output file
                                    init_model_filesuffix = "_historical",    # if you want to start from a previous model run state
                                    climate_filename = 'gcm_data',
                                    mb_type="mb_real_daily",                  # use daily temperature values
                                    grad_type="cte",                          # constant lapse rate (-6.5)
                                    precipitation_factor=1,                   # multiply a factor to the precipitation time series
                                    melt_f='from_json');                      # calibrated melt_f (changed climate_type)

    utils.compile_run_output(gdirs, input_filesuffix= "_" + rid);

## What-if runs

In [ ]:
scenarios  = glob(config["dir"]["data"] + "/cmip6/*.nc")

for gcm in tqdm(scenarios):    

    rid = os.path.splitext(os.path.basename(gcm))[0].rsplit('_', 2)[0]
    
    workflow.execute_entity_task(run_with_hydro_daily, gdirs,
                                run_task = run_from_climate_data_TIModel, # any of the `run_*`` tasks in the MBSandbox.flowline_TIModel module.
                                ref_area_from_y0 = True,                  # overwrite ref_area_yr to the first year of the timeseries
                                ref_geometry_filesuffix = "_historical",  # this kwarg allows to copy the reference area from a previous simulation
                                store_annual = True,                      # whether to store annual outputs or only daily outputs
                                mb_type="mb_real_daily",                  # use daily temperature values
                                grad_type="cte",                          # constant lapse rate (-6.5)
                                precipitation_factor=1,                   # multiply a factor to the precipitation time series
                                melt_f='from_json',                       # calibrated melt_f (changed climate_type)
                                ys=2000, ye=2025,                         # start and end years of the model run
                                climate_input_filesuffix= "ERA5",         # filesuffix for the input climate file                        
                                init_model_filesuffix =  "_"+  rid ,      # if you want to start from a previous model run state
                                output_filesuffix= "_historical_" +  rid, # for the output file
                                climate_filename = 'climate_historical');
    
    # init_model_yr = 2050 if you want to start from a previous model run state (not the last year)
    utils.compile_run_output(gdirs, input_filesuffix= "_historical_" +  rid);